In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sessions = pd.read_csv('../data_csv/3 sessions_cleaned.csv')
print(sessions.shape)

In [ ]:
# build funnel
has_view     = sessions['num_views'] > 0
has_cart     = sessions['num_cart'] > 0
has_purchase = sessions['num_purchase'] > 0

funnel = pd.Series({
    'View':     has_view.sum(),
    'Cart':     (has_view & has_cart).sum(),        # 有浏览且有加购
    'Purchase': (has_view & has_cart & has_purchase).sum()  # 三个都有
})

print(funnel)
print(f"View → Cart:     {funnel['Cart']/funnel['View']:.2%}")
print(f"Cart → Purchase: {funnel['Purchase']/funnel['Cart']:.2%}")
print(f"Overall:         {funnel['Purchase']/funnel['View']:.2%}")

View        4280699
Cart         788430
Purchase     106526
dtype: int64
View → Cart:     18.42%
Cart → Purchase: 13.51%
Overall:         2.49%


In [31]:
sessions["converted"].value_counts()
convert_rate=155617/4380323
print(f'{convert_rate:.2%}')

3.55%


Create new features

In [35]:
sessions.columns

Index(['user_session', 'user_id', 'session_start', 'session_end',
       'total_events', 'num_views', 'num_cart', 'num_purchase', 'num_remove',
       'num_returns', 'unique_products', 'avg_price', 'max_price', 'hour',
       'day_of_week', 'session_duration_min', 'converted', 'view_depth',
       'single_view', 'hesitation_rate', 'cart_abandoned'],
      dtype='object')

In [34]:
#view to cart
sessions['view_depth'] = sessions['unique_products'] / sessions['num_views']
sessions['single_view'] = (sessions['num_views'] == 1).astype(int)

# cart to purchase
sessions['hesitation_rate'] = sessions['num_remove'] / sessions['num_cart'].replace(0, np.nan)
sessions['cart_abandoned'] = ((sessions['num_cart'] > 0) & (sessions['num_purchase'] == 0)).astype(int)

print(sessions[['view_depth','single_view','hesitation_rate','cart_abandoned']].describe())

c:\Users\sungu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


         view_depth   single_view  hesitation_rate  cart_abandoned
count  4.535940e+06  4.535940e+06    985780.000000    4.535940e+06
mean            inf  6.944259e-01         0.517730    1.894846e-01
std             NaN  4.606502e-01         2.495554    3.918932e-01
min    6.578947e-03  0.000000e+00         0.000000    0.000000e+00
25%    1.000000e+00  0.000000e+00         0.000000    0.000000e+00
50%    1.000000e+00  1.000000e+00         0.000000    0.000000e+00
75%    1.000000e+00  1.000000e+00         0.500000    0.000000e+00
max             inf  1.000000e+00      1048.000000    1.000000e+00


In [45]:
sessions.head(2)

,user_session,user_id,session_start,session_end,total_events,num_views,num_cart,num_purchase,num_remove,num_returns,...,avg_price,max_price,hour,day_of_week,session_duration_min,converted,view_depth,single_view,hesitation_rate,cart_abandoned
0,0000061d-f3e9-484b-8c73-e54f355032a3,539262914,2020-01-16 06:30:41+03:00,2020-01-16 06:30:41+03:00,1,1,0,0,0,0,...,194.44,194.44,6,Thursday,0.0,0,1.0,1,NaN,0
1,00000ac8-0015-4f12-996a-be2896323738,605114412,2020-01-25 01:22:20+03:00,2020-01-25 01:22:20+03:00,1,1,0,0,0,0,...,25.71,25.71,1,Saturday,0.0,0,1.0,1,NaN,0


In [ ]:
# check remove to  cart session
weird_hesitation = sessions[sessions['num_remove'] > sessions['num_cart']]
print(f"count: {len(weird_hesitation)}")
print(weird_hesitation[['num_views','num_cart','num_purchase','num_remove']].head(10))

数量: 198477
     num_views  num_cart  num_purchase  num_remove
3            2         1             9           8
16           0         5             0           9
40           1         6             4          11
58           0         0             0           4
61          16         2             0           4
81           2         1             0           4
131          0         0             0           1
139          1         0             0           2
182         26         3             0          26
192         26         4             0           7


In [48]:
sessions['view_depth'] = sessions['unique_products'] / sessions['num_views'].replace(0, np.nan)
sessions['view_depth'] = sessions['view_depth'].replace([np.inf, -np.inf], np.nan)
sessions['hesitation_rate'] = sessions['hesitation_rate'].clip(0, 1)

print(sessions[['view_depth','hesitation_rate']].describe())

         view_depth  hesitation_rate
count  4.280699e+06    985780.000000
mean   1.315175e+00         0.255042
std    2.605095e+00         0.379149
min    6.578947e-03         0.000000
25%    1.000000e+00         0.000000
50%    1.000000e+00         0.000000
75%    1.000000e+00         0.500000
max    1.080000e+03         1.000000


In [51]:
sessions['view_depth'].isnull().mean()

np.float64(0.056270806051226425)

In [ ]:
print(sessions['view_depth'].quantile([0.95, 0.99, 0.999]))

0.950     2.5
0.990     9.0
0.999    30.0
Name: view_depth, dtype: float64


In [14]:
sessions['view_depth'] = sessions['view_depth'].clip(0, 30)
print(sessions['view_depth'].describe())

count    4.280699e+06
mean     1.292592e+00
std      1.811573e+00
min      6.578947e-03
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+01
Name: view_depth, dtype: float64


In [ ]:

sessions.to_csv('sessions_featured.csv', index=False)